# Phase 2 — Section 2 Report
## EDA, Data Preprocessing, and Feature Engineering

**Dataset:** Crawled Stack Overflow C++ Questions Dataset  
**Final task:** Semantic similar-question recommendation  

Run `python pipeline.py` before this notebook. The cells below query the SQLite database and inspect the same data that is used by the pipeline.

## 1. Preparation scope

Section 1 handled **ingestion integrity**: source-field validation, unique question IDs, valid tag lists, normalized tables, and timestamp conversion. This section handles **modeling preparation**: EDA, missingness, noisy fields, text cleaning, distribution/outlier checks, and feature engineering. Section 3 packages the same steps into modular, reproducible scripts.

In [1]:
from pathlib import Path
import os
import sys

import matplotlib
matplotlib.use('Agg')  # Non-interactive backend: works consistently in VS Code and automated runs.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
environment_root = Path(os.environ['PROJECT_ROOT']).resolve() if os.environ.get('PROJECT_ROOT') else None
candidates = [path for path in [environment_root, cwd, *cwd.parents, cwd / 'Phase 2' / 'Project_P2'] if path is not None]
PROJECT_ROOT = next(
    (path for path in candidates if (path / 'pipeline.py').exists() and (path / 'requirements.txt').exists()),
    cwd,
)
sys.path.insert(0, str(PROJECT_ROOT))

from scripts.database_connection import get_database_path
from scripts.load_data import load_question_dataframe

DATABASE_PATH = get_database_path()
if not DATABASE_PATH.exists():
    raise FileNotFoundError(f'Database not found: {DATABASE_PATH}. Run python pipeline.py first.')

questions = load_question_dataframe()
questions['creation_at'] = pd.to_datetime(questions['creation_at'], utc=True, errors='coerce')
questions['last_activity_at'] = pd.to_datetime(questions['last_activity_at'], utc=True, errors='coerce')
print(f'Project root: {PROJECT_ROOT}')
print(f'Questions loaded from SQLite: {len(questions):,}')

Project root: /home/user/Data-Science/Phase 2/Project_P2
Questions loaded from SQLite: 2,500


## 2. Dataset structure

The recommendation input is the title plus the HTML body. Tags and engagement variables are retained as metadata; owner profile and sparse migration fields are deliberately excluded from modeling because they do not represent the technical meaning of a question.

In [2]:
summary = pd.DataFrame({
    'column': questions.columns,
    'dtype': questions.dtypes.astype(str).values,
    'missing_values': questions.isna().sum().values,
    'unique_values': questions.nunique(dropna=True).values,
})
display(summary)
display(questions.head(3))

,column,dtype,missing_values,unique_values
0,question_id,int64,0,2500
1,title,str,0,2500
2,body_html,str,0,2500
3,question_url,str,0,2500
4,is_answered,int64,0,2
5,view_count,int64,0,1550
6,answer_count,int64,0,37
7,score,int64,0,195
8,accepted_answer_id,float64,1250,1250
9,creation_at,"datetime64[us, UTC]",0,2500


,question_id,title,body_html,question_url,is_answered,view_count,answer_count,score,accepted_answer_id,creation_at,last_activity_at,last_edit_at,closed_at,closed_reason,tags,tag_count
0,25,How to use the C socket API in C++ on z/OS,<p>I'm having issues getting the C sockets API...,https://stackoverflow.com/questions/25/how-to-...,1,17477,9,180,1443907.0,2008-08-01 12:13:50+00:00,2025-12-16 13:40:20+00:00,2023-06-01T11:20:00Z,NaN,NaN,c|c++|mainframe|sockets|zos,5
1,17333,How do you compare float and double while acco...,\n<p>What would be the most efficient way to c...,https://stackoverflow.com/questions/17333/how-...,1,693784,32,680,NaN,2008-08-20 02:09:33+00:00,2026-02-03 17:51:16+00:00,2023-09-18T06:20:59Z,NaN,NaN,algorithm|c++|floating-point|optimization,4
2,39419,How large is a DWORD with 32- and 64-bit code?,<p>In Visual C++ a DWORD is just an unsigned l...,https://stackoverflow.com/questions/39419/how-...,1,168399,4,65,39441.0,2008-09-02 12:50:26+00:00,2026-04-13 18:08:29+00:00,2016-09-30T14:26:16Z,NaN,NaN,64-bit|c++|dword|winapi,4


## 3. Missing values and data quality

Rows without title or body cannot be used for semantic retrieval and are removed by `preprocess.py`. Optional dates and accepted-answer IDs are not imputed because their absence has a natural meaning (for example, an unedited question has no edit date). Tags are normalized and an empty tag list would be explicitly recorded.

In [3]:
quality_checks = pd.DataFrame({
    'check': [
        'duplicate question_id', 'missing title', 'missing body_html',
        'blank title', 'blank body_html', 'questions without tags'
    ],
    'count': [
        questions['question_id'].duplicated().sum(),
        questions['title'].isna().sum(),
        questions['body_html'].isna().sum(),
        questions['title'].fillna('').str.strip().eq('').sum(),
        questions['body_html'].fillna('').str.strip().eq('').sum(),
        questions['tags'].isna().sum(),
    ],
})
display(quality_checks)
display(questions.isna().sum().sort_values(ascending=False).to_frame('missing_values'))

,check,count
0,duplicate question_id,0
1,missing title,0
2,missing body_html,0
3,blank title,0
4,blank body_html,0
5,questions without tags,0


,missing_values
closed_at,2256
closed_reason,2256
accepted_answer_id,1250
last_edit_at,567
question_id,0
title,0
body_html,0
question_url,0
score,0
answer_count,0


## 4. Numeric distributions and outliers

Views, answers, and score measure community engagement rather than semantic similarity. They are highly skewed, so the pipeline keeps their raw versions for auditability but creates `log1p`/signed-log transformations and standardized variants for optional ranking metadata. Raw values are not used as the text-similarity target.

In [4]:
numeric_columns = ['view_count', 'answer_count', 'score', 'tag_count']
display(questions[numeric_columns].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T)

def iqr_outlier_count(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return int(((series < q1 - 1.5 * iqr) | (series > q3 + 1.5 * iqr)).sum())

outliers = pd.DataFrame({
    'feature': numeric_columns,
    'iqr_outlier_count': [iqr_outlier_count(questions[column]) for column in numeric_columns],
})
display(outliers)

,count,mean,std,min,1%,25%,50%,75%,99%,max
view_count,2500.0,25127.6464,156232.265188,10.0,58.0,171.0,500.5,6067.75,457964.26,5131807.0
answer_count,2500.0,3.5972,4.841521,0.0,0.0,1.0,2.0,4.00,25.00,52.0
score,2500.0,35.3344,563.175400,-25.0,-3.0,0.0,2.0,7.00,486.43,27530.0
tag_count,2500.0,3.4392,1.146311,1.0,1.0,3.0,3.0,4.00,5.00,5.0


,feature,iqr_outlier_count
0,view_count,382
1,answer_count,265
2,score,367
3,tag_count,120


In [5]:
plot_data = questions.copy()
plot_data['log_view_count'] = np.log1p(plot_data['view_count'].clip(lower=0))
plot_data['signed_log_score'] = np.sign(plot_data['score']) * np.log1p(plot_data['score'].abs())
monthly_questions = plot_data.set_index('creation_at').resample('ME').size()

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes[0, 0].hist(plot_data['log_view_count'], bins=40, color='#377eb8')
axes[0, 0].set(title='log1p(view count)', xlabel='log1p(views)', ylabel='questions')
axes[0, 1].hist(plot_data['signed_log_score'], bins=40, color='#4daf4a')
axes[0, 1].set(title='Signed log score', xlabel='signed log1p(score)', ylabel='questions')
axes[1, 0].hist(plot_data['tag_count'], bins=np.arange(0.5, 6.5, 1), color='#984ea3', rwidth=0.9)
axes[1, 0].set(title='Tags per question', xlabel='tag count', ylabel='questions')
axes[1, 1].plot(monthly_questions.index, monthly_questions.values, color='#e41a1c', linewidth=1)
axes[1, 1].set(title='Question creation over time', xlabel='month', ylabel='questions')
fig.tight_layout()
display(fig)
plt.close(fig)

<Figure size 1400x900 with 4 Axes>

## 5. Text and tag patterns

The corpus is entirely within the C++ domain, but secondary tags reveal subtopics. This supports a two-stage future retrieval design: semantic similarity over the cleaned document text, optionally constrained or reranked using tags.

In [6]:
tag_frequency = (
    questions['tags'].fillna('').str.split('|').explode().replace('', pd.NA).dropna()
    .value_counts().rename_axis('tag').reset_index(name='question_count')
)
display(tag_frequency.head(20))

text_lengths = pd.DataFrame({
    'title_characters': questions['title'].fillna('').str.len(),
    'body_html_characters': questions['body_html'].fillna('').str.len(),
})
display(text_lengths.describe(percentiles=[0.01, 0.5, 0.95, 0.99]).T)

,tag,question_count
0,c++,2500
1,c,128
2,c++11,117
3,qt,105
4,windows,104
5,templates,102
6,cmake,101
7,language-lawyer,98
8,winapi,89
9,c++20,89


,count,mean,std,min,1%,50%,95%,99%,max
title_characters,2500.0,61.0748,24.026588,15.0,21.00,57.0,107.0,131.00,160.0
body_html_characters,2500.0,1980.5688,2549.907916,41.0,140.99,1263.5,5905.4,14577.63,26117.0


## 6. Implemented preprocessing and feature engineering

`scripts/preprocess.py` removes duplicate/invalid text rows, turns HTML into normalized text, normalizes tag lists, parses dates, and records all decisions in `data/reports/preprocessing_report.json`.

`scripts/feature_engineering.py` then creates title/body/document word counts, unique-token ratio, code-block count, creation year/month/age, tag count, log-transformed engagement values, standardized (z-score) metadata, **multi-hot indicator columns for the most frequent informative tags** (categorical encoding), and TF-IDF unigrams/bigrams.

**What feeds the model vs. what is kept for auditing.** The report's `model_text_input` (the TF-IDF matrix) plus `model_numeric_features` (the `*_zscore` columns and the `tag_*` multi-hot columns) are the modeling inputs. Raw and intermediate columns listed under `audit_only_columns` (e.g. raw `view_count`, `score`, and the pre-standardization `*_log1p` values) are deliberately retained in the CSV for traceability but are **not** fed to the model, because keeping both the raw and standardized versions would introduce near-perfectly correlated duplicate features. The age feature is measured against the dataset's most recent question (`age_reference_date`) so the pipeline stays deterministic across reruns.

The TF-IDF matrix is the direct feature representation for cosine-similarity recommendation.

In [7]:
import json

report_dir = PROJECT_ROOT / 'data' / 'reports'
for report_name in ['preprocessing_report.json', 'feature_engineering_report.json']:
    report_path = report_dir / report_name
    if report_path.exists():
        print(f'\n{report_name}')
        display(pd.Series(json.loads(report_path.read_text(encoding='utf-8'))))
    else:
        print(f'{report_name} is not available yet; run python pipeline.py.')


preprocessing_report.json


input_rows                                                                         2500
duplicate_question_ids_removed                                                        0
missing_or_blank_core_text_removed                                                    0
empty_documents_removed                                                               0
missing_tag_lists                                                                     0
numeric_missing_after_coercion        {'view_count': 0, 'answer_count': 0, 'score': ...
numeric_correlation_matrix            {'view_count': {'view_count': 1.0, 'answer_cou...
high_correlation_pairs_abs_ge_0_90                                                   []
outlier_strategy                      Retain legitimate high-engagement questions, t...
output_rows                                                                        2500
excluded_from_similarity_model        [question_url, accepted_answer_id, closed_reas...
dtype: object


feature_engineering_report.json


rows                                                                     2500
tfidf_shape                                                     [2500, 12000]
tfidf_vocabulary_size                                                   12000
tfidf_settings              {'ngram_range': [1, 2], 'min_df': 2, 'max_df':...
age_reference_date                                  2026-05-28T16:11:59+00:00
scaled_metadata_features    [view_count_log1p, answer_count_log1p, score_s...
tag_multihot_columns        [tag_c, tag_c_11, tag_qt, tag_windows, tag_tem...
encoded_tags                [c, c++11, qt, windows, templates, cmake, lang...
model_text_input            data/models/question_tfidf_matrix.npz (TF-IDF ...
model_numeric_features      [view_count_log1p_zscore, answer_count_log1p_z...
audit_only_columns          [view_count, answer_count, score, view_count_l...
similarity_input            TF-IDF vectors of title + cleaned body; cosine...
dtype: object

## 7. Section 2 conclusion

The data is ready for direct modeling: every retained row has usable question text, the preprocessing is deterministic, and the feature artefacts are saved. The next stage uses `pipeline.py` to reproduce these results automatically; Phase 3 can use the saved TF-IDF matrix to compute cosine similarity between a new question and historical questions.